# MCP Tools Demo

Shows how to expose MCP tools via the registry and run them through `ToolCapability`.

In [ ]:
import os
from getpass import getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, MCPServerConfig, MCPToolDescriptor, MCPToolRegistry, FunctionAdapter
from agentic_codex.core.schemas import AgentStep, Message

# Build a stub MCP registry
server = MCPServerConfig(name="local", base_url="https://mcp.local")
registry = MCPToolRegistry(server)
registry.register(MCPToolDescriptor(name="search.docs", description="Search docs", path="/search"))
registry.register(MCPToolDescriptor(name="lookup.user", description="User lookup", path="/users"))

def stub_llm(prompt: str) -> str:
    return f"[stub llm] {prompt[:80]}"

def step(ctx: Context) -> AgentStep:
    search = ctx.get_tool("search.docs")
    lookup = ctx.get_tool("lookup.user")
    res1 = search.invoke(query="policies")
    res2 = lookup.invoke(user_id=123)
    content = f"search={res1['transport']} lookup={res2['transport']}"
    return AgentStep(out_messages=[Message(role="mcp-agent", content=content)], state_updates={"mcp": res1})

agent = (
    AgentBuilder(name="mcp-agent", role="operator")
    .with_capability(registry.as_capability())
    .with_llm(FunctionAdapter(stub_llm))
    .with_step(step)
    .build()
)

ctx = Context(goal="use mcp")
out = agent.run(ctx)
out.out_messages[-1].content